In [ ]:
# Only import/initialize chromadb and chroma_client if they are not already defined
if 'chromadb' not in globals():
	import chromadb

if 'chroma_client' not in globals():
	chroma_client = chromadb.PersistentClient(path="./ChromaDb")

In [ ]:
# Reuse existing collection if it already exists, otherwise create it.
existing = next((c for c in chroma_client.list_collections() if c.name == "collection1"), None)
if existing is not None:
	collection = existing
else:
	collection = chroma_client.create_collection(name="collection1")

In [ ]:
collection.add(
    ids=["1", "2", "3"],
    embeddings=[[0.1, 0.2, 0.3], [0.2, 0.3, 0.4], [0.3, 0.4, 0.5]],
    metadatas=[{"name": "Prakash"}, {"name": "Aradhana"}, {"name": "Techy Prakash"}]
    )

In [5]:
print("Available Collections:", chroma_client.list_collections())

Available Collections: [Collection(name=collection1), Collection(name=collection3)]


In [6]:
print("Data with ID 2:", collection.get(ids=["2"], include=["metadatas", "embeddings"]))

Data with ID 2: {'ids': ['2'], 'embeddings': array([[0.2       , 0.30000001, 0.40000001]]), 'documents': None, 'uris': None, 'included': ['metadatas', 'embeddings'], 'data': None, 'metadatas': [{'name': 'Aradhana'}]}


In [7]:
collection2 = chroma_client.create_collection(name="collection2")

In [8]:
collection2.add(
    ids=["4"],
    embeddings=[0.4, 0.5, 0.6],
    documents=["New document added for Anay"]
    )

In [9]:
collection2.add(
    ids=["5"],
    embeddings=[0.4, 0.5, 0.6],
    documents=["New document added for Papa"]
    )

In [10]:
collection2.get(ids=["4"], include=["metadatas", "documents", "embeddings"])

{'ids': ['4'],
 'embeddings': array([[0.40000001, 0.5       , 0.60000002]]),
 'documents': ['New document added for Anay'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [None]}

In [11]:
print("Available Collections:", chroma_client.list_collections())

Available Collections: [Collection(name=collection1), Collection(name=collection3), Collection(name=collection2)]


In [12]:
collection2.update(
    ids=["4"], 
    embeddings=[0.5, 0.6, 0.7],
    documents=["Updated document for Anay"])

In [13]:
collection2.get(ids=["4"], include=["metadatas", "documents", "embeddings"])

{'ids': ['4'],
 'embeddings': array([[0.5       , 0.60000002, 0.69999999]]),
 'documents': ['Updated document for Anay'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [None]}

In [14]:
collection2.get(ids=["5"], include=["metadatas", "documents", "embeddings"])

{'ids': ['5'],
 'embeddings': array([[0.40000001, 0.5       , 0.60000002]]),
 'documents': ['New document added for Papa'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [None]}

In [15]:
collection2.delete(ids=["5"])

In [16]:
collection2.get(ids=["5"], include=["metadatas", "documents", "embeddings"])

{'ids': [],
 'embeddings': array([], dtype=float64),
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': []}

In [17]:
print("Available Collections:", chroma_client.list_collections())

Available Collections: [Collection(name=collection1), Collection(name=collection3), Collection(name=collection2)]


In [18]:
#chroma_client.delete_collection(name="collection2")
for coll in chroma_client.list_collections():
    chroma_client.delete_collection(name=coll.name)

In [19]:
print("Available Collections:", chroma_client.list_collections())

Available Collections: []


In [21]:
#Open AI Embedding Model

In [20]:
from dotenv import load_dotenv
load_dotenv()
import os
api_key =os.getenv("GOOGLE_API_KEY")

In [21]:
chroma_client = chromadb.PersistentClient(path="./ChromaDb")

In [22]:
collection3 = chroma_client.create_collection(name="collection3")

In [23]:
from google import genai
client = genai.Client(api_key=api_key)
def get_googleai_embedding(text):
    response = client.models.embed_content(model="gemini-embedding-001",contents=[text])
    return response.embeddings[0]

In [24]:
documents= [
    "The Eiffel Tower is located in Paris.",
    "The Colosseum is an ancient amphitheater in Rome.",
    "The Taj Mahal is a mausoleum in Agra, India.",
    "Mount Everest is the highest mountain in the world.",
    "Python is a popular programming language."
]

In [25]:
#coverting documents to embeddings
embeddings = [get_googleai_embedding(doc) for doc in documents]

In [26]:
print("embedding",embeddings) #vector size 3072

embedding [ContentEmbedding(
  values=[
    -0.014392886,
    0.023128558,
    0.020589013,
    -0.05267376,
    -0.040127832,
    <... 3067 more items ...>,
  ]
), ContentEmbedding(
  values=[
    -0.025581352,
    0.003487143,
    -0.011464868,
    -0.061934225,
    -0.031244392,
    <... 3067 more items ...>,
  ]
), ContentEmbedding(
  values=[
    -0.033939876,
    -0.008885777,
    -0.0046633943,
    -0.06804953,
    0.012996041,
    <... 3067 more items ...>,
  ]
), ContentEmbedding(
  values=[
    -0.01745653,
    0.02275,
    -0.0032639815,
    -0.048338555,
    -0.0067085903,
    <... 3067 more items ...>,
  ]
), ContentEmbedding(
  values=[
    -0.011742769,
    -0.008342069,
    9.026351e-05,
    -0.046162803,
    -0.024655225,
    <... 3067 more items ...>,
  ]
)]


In [27]:
#len(embeddings)
len(documents)

5

In [28]:
import numpy as np
embedding_dimension = 3072

embeddings = [np.random.rand(embedding_dimension).tolist() for _ in range(len(documents))]

embeddings_np_array = np.array(embeddings)

print("\n--- Final Pre-Add Diagnostic ---")
print(f"Variable being passed as 'embeddings': {type(embeddings_np_array)}")
print(f"Shape of array being passed: {embeddings_np_array.shape}")
print(f"Data type (dtype) of array elements: {embeddings_np_array.dtype}")
print(f"Is it a NumPy array? {isinstance(embeddings_np_array, np.ndarray)}")
print(f"First element of the array (first 10 values): {embeddings_np_array[0][:10]}...")


--- Final Pre-Add Diagnostic ---
Variable being passed as 'embeddings': <class 'numpy.ndarray'>
Shape of array being passed: (5, 3072)
Data type (dtype) of array elements: float64
Is it a NumPy array? True
First element of the array (first 10 values): [0.77304685 0.10818519 0.60975933 0.28819985 0.316035   0.14600525
 0.89790302 0.41811535 0.61539387 0.97271536]...


In [29]:
collection3.add(
    ids=[str(i) for i in range(len(documents))],
    embeddings=embeddings_np_array, # Pass the 2D NumPy array here
    documents=documents
)

In [31]:
query_text = "Where is the Eiffel Tower located?"
query_embedding_object  = get_googleai_embedding(query_text)
query_embedding_values = query_embedding_object.values # Extract the numerical values
results = collection3.query(
    query_embeddings=[query_embedding_values],
    n_results=3,
    include=["documents", "distances"]
)
print("Query:", query_text)
print("Most similar result:", results["documents"][0])
print("Distances:", results["distances"][0])

Query: Where is the Eiffel Tower located?
Most similar result: ['Python is a popular programming language.', 'The Taj Mahal is a mausoleum in Agra, India.', 'The Colosseum is an ancient amphitheater in Rome.']
Distances: [990.0682983398438, 995.843017578125, 1007.19287109375]


In [32]:
updated_text="The eiffel tower is one of the most visited landmarks in the world."
updated_embedding_object  = get_googleai_embedding(updated_text)
updated_embedding_values = updated_embedding_object.values # Extract the numerical values

collection3.update(
    ids=["0"], 
    embeddings=[updated_embedding_values],
    documents=[updated_text]
)

In [33]:
query_text = "Where is the Eiffel Tower located?"
query_embedding_object  = get_googleai_embedding(query_text)
query_embedding_values = query_embedding_object.values # Extract the numerical values
results = collection3.query(
    query_embeddings=[query_embedding_values], n_results=2, include=["documents", "distances"]
)
print("Query:", query_text)
print("Most similar result:", results["documents"][0])
print("Distances:", results["distances"][0])

Query: Where is the Eiffel Tower located?
Most similar result: ['The eiffel tower is one of the most visited landmarks in the world.', 'Python is a popular programming language.']
Distances: [0.5706296563148499, 990.0682983398438]


In [34]:
query_text = "C# is my favorite programming language."
query_embedding_object  = get_googleai_embedding(query_text)
query_embedding_values = query_embedding_object.values # Extract the numerical values
results = collection3.query(
    query_embeddings=[query_embedding_values], n_results=3, include=["documents", "distances"]
)
print("Query:", query_text)
print("Most similar result:", results["documents"][0])
print("Distances:", results["distances"][0])

Query: C# is my favorite programming language.
Most similar result: ['The eiffel tower is one of the most visited landmarks in the world.', 'Python is a popular programming language.', 'The Taj Mahal is a mausoleum in Agra, India.']
Distances: [0.9812070727348328, 990.6468505859375, 997.4769897460938]
